## Step 1: Mount Drive & Restore Paths from Day 1

In [ ]:
# Mount Drive (same as Day 1)
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Folder paths — same structure created on Day 1
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {
    "raw":           os.path.join(BASE_DIR, "data/raw"),
    "processed":     os.path.join(BASE_DIR, "data/processed"),
    "metadata":      os.path.join(BASE_DIR, "data/metadata"),
    "notebooks":     os.path.join(BASE_DIR, "notebooks"),
    "visualizations":os.path.join(BASE_DIR, "visualizations")
}

print('Drive mounted ✅')
print('Paths restored:')
for k, v in DIRS.items():
    print(f'  {k}: {v}')

## Step 2: Define the Bounding Box
Same values as Day 0 geographic exploration. Defining them here again so this notebook is self-contained.

In [ ]:
# North Indian Ocean Bounding Box
LAT_MIN =  5   # 5°N
LAT_MAX = 30   # 30°N
LON_MIN = 65   # 65°E
LON_MAX = 95   # 95°E

print(f'Bounding Box → Lat: {LAT_MIN}°N – {LAT_MAX}°N | Lon: {LON_MIN}°E – {LON_MAX}°E')
print('Covers: Arabian Sea | Bay of Bengal | Lakshadweep Sea | Andaman Sea')

## Step 3: Load All 11 Raw Datasets

In [ ]:
# --- Dataset 1: Primary Plastic Pollution Data ---
# Date format: '01/01/2015 0:00' → day first, needs dayfirst=True
df_plastic = pd.read_csv(
    os.path.join(DIRS['raw'], 'ocean_plastic_pollution_data.csv'),
    parse_dates=['Date'],
    dayfirst=True
)
print(f'[1] ocean_plastic_pollution_data → {df_plastic.shape[0]:,} rows × {df_plastic.shape[1]} cols')
print(f'    Columns: {df_plastic.columns.tolist()}')
print(f'    Date range: {df_plastic["Date"].min().date()} to {df_plastic["Date"].max().date()}')

In [ ]:
# --- Dataset 2: Primary Marine Species Data ---
# event_date has mixed formats ('2023-02-24T17:21:58', '2003-02-24T16:05', '2022-10-14')
# Using errors='coerce' so malformed dates become NaT instead of crashing
df_species = pd.read_csv(os.path.join(DIRS['raw'], 'india_cms_final_master_enriched.csv'))
df_species['event_date'] = pd.to_datetime(df_species['event_date'], errors='coerce')

print(f'[2] india_cms_final_master_enriched → {df_species.shape[0]:,} rows × {df_species.shape[1]} cols')
print(f'    Columns: {df_species.columns.tolist()}')
print(f'    Taxonomic groups: {df_species["taxonomic_group"].value_counts().to_dict()}')
# Flag the year anomaly spotted in Day 0
print(f'    ⚠️  Year range: {df_species["year"].min()} to {df_species["year"].max()} — anomalies present')

In [ ]:
# --- Dataset 3: Realistic Ocean Climate (SST + Bleaching + Species Observed) ---
df_climate = pd.read_csv(
    os.path.join(DIRS['raw'], 'realistic_ocean_climate_dataset.csv'),
    parse_dates=['Date']
)
print(f'[3] realistic_ocean_climate_dataset → {df_climate.shape[0]:,} rows × {df_climate.shape[1]} cols')
print(f'    Locations: {df_climate["Location"].unique().tolist()}')
print(f'    Date range: {df_climate["Date"].min().date()} to {df_climate["Date"].max().date()}')

In [ ]:
# --- Dataset 4: Global Temperatures ---
df_temp = pd.read_csv(
    os.path.join(DIRS['raw'], 'GlobalTemperatures.csv'),
    parse_dates=['dt']
)
print(f'[4] GlobalTemperatures → {df_temp.shape[0]:,} rows × {df_temp.shape[1]} cols')
print(f'    Date range: {df_temp["dt"].min().date()} to {df_temp["dt"].max().date()}')
print(f'    No lat/lon in this dataset — global averages only')

In [ ]:
# --- Dataset 5: Sea Level + CO2 ---
df_co2 = pd.read_csv(
    os.path.join(DIRS['raw'], 'sea_level_co2.csv'),
    parse_dates=['date']
)
print(f'[5] sea_level_co2 → {df_co2.shape[0]:,} rows × {df_co2.shape[1]} cols')
print(f'    Year range: {df_co2["year"].min()} to {df_co2["year"].max()}')
print(f'    No lat/lon — global time-series only')

In [ ]:
# --- Dataset 6: Country-Level Plastic Waste ---
df_world = pd.read_csv(os.path.join(DIRS['raw'], 'Plastic_Waste_Around_the_World.csv'))
print(f'[6] Plastic_Waste_Around_the_World → {df_world.shape[0]:,} rows × {df_world.shape[1]} cols')
print(f'    Countries: {df_world.shape[0]} | No lat/lon — country-level aggregates')
print(f'    India row present: {"India" in df_world["Country"].values}')

In [ ]:
# --- Dataset 7: River Plastic Risk Scenarios ---
df_river = pd.read_csv(os.path.join(DIRS['raw'], 'River_Plastic_Waste_Risk_Scenarios_2015_2060.csv'))
print(f'[7] River_Plastic_Waste_Risk_Scenarios → {df_river.shape[0]:,} rows × {df_river.shape[1]} cols')
print(f'    India rivers: {len(df_river[df_river["Country"]=="India"])}')
print(f'    No lat/lon — river-level, will join via Country key')

In [ ]:
# --- Dataset 8: India Protected Sites ---
df_protected = pd.read_csv(os.path.join(DIRS['raw'], 'protected_sites_india.csv'))
print(f'[8] protected_sites_india → {df_protected.shape[0]:,} rows × {df_protected.shape[1]} cols')
print(f'    Types: {df_protected["Type"].value_counts().to_dict()}')

In [ ]:
# --- Datasets 9, 10, 11: Microplastic Surveys ---
df_micro_adv = pd.read_csv(
    os.path.join(DIRS['raw'], 'ADVENTURE_MICRO_FROM_SCIENTIST.csv'),
    parse_dates=['Date']
)
df_micro_geo = pd.read_csv(
    os.path.join(DIRS['raw'], 'GEOMARINE_MICRO.csv'),
    parse_dates=['Date']
)
df_micro_sea = pd.read_csv(
    os.path.join(DIRS['raw'], 'SEA_MICRO.csv'),
    parse_dates=['Date']
)

print(f'[9]  ADVENTURE_MICRO → {df_micro_adv.shape[0]:,} rows | Unit: Total_Pieces_L (pieces/litre)')
print(f'[10] GEOMARINE_MICRO → {df_micro_geo.shape[0]:,} rows | Unit: MP_conc (particles/m³)')
print(f'[11] SEA_MICRO       → {df_micro_sea.shape[0]:,} rows | Unit: Pieces_KM2')
print(f'     ⚠️  All 3 use different units — use Normalized column only for spatial overlay')

In [ ]:
# Summary: all 11 datasets loaded
all_datasets = {
    'ocean_plastic':    df_plastic,
    'india_species':    df_species,
    'ocean_climate':    df_climate,
    'global_temp':      df_temp,
    'sea_level_co2':    df_co2,
    'plastic_world':    df_world,
    'river_plastic':    df_river,
    'protected_sites':  df_protected,
    'micro_adventure':  df_micro_adv,
    'micro_geomarine':  df_micro_geo,
    'micro_sea':        df_micro_sea,
}

print('\n' + '='*50)
print('ALL 11 DATASETS LOADED SUCCESSFULLY ✅')
print('='*50)
print(f'{"Name":<20} {"Rows":>8} {"Cols":>6}')
print('-'*38)
for name, df in all_datasets.items():
    print(f'{name:<20} {df.shape[0]:>8,} {df.shape[1]:>6}')

---
## Step 4: Apply Bounding Box Filter to Spatial Datasets

Only datasets with `Latitude` and `Longitude` columns get filtered.
The others (GlobalTemperatures, sea_level_co2, etc.) have no coordinates — they will be joined later by year/month keys.

> 💡 **Reusable helper:** I'll write one function and apply it to every spatial dataset so the logic is consistent everywhere.

In [ ]:
def apply_bbox_filter(df, lat_col, lon_col, label=''):
    """
    Filter a DataFrame to the North Indian Ocean bounding box.
    Returns the filtered DataFrame and prints a summary.
    """
    mask = (
        (df[lat_col] >= LAT_MIN) &
        (df[lat_col] <= LAT_MAX) &
        (df[lon_col] >= LON_MIN) &
        (df[lon_col] <= LON_MAX)
    )
    filtered = df[mask].copy()
    pct = round(len(filtered) / len(df) * 100, 1)
    print(f'{label:<25} {len(df):>8,} → {len(filtered):>6,} records retained  ({pct}%)')
    return filtered

print(f'Bounding Box: Lat {LAT_MIN}°N–{LAT_MAX}°N | Lon {LON_MIN}°E–{LON_MAX}°E')
print('='*70)
print(f'{"Dataset":<25} {"Before":>8}   {"After":>6}   Retained')
print('-'*70)

In [ ]:
# Apply to primary plastic dataset
df_plastic_filtered = apply_bbox_filter(df_plastic, 'Latitude', 'Longitude', 'ocean_plastic')

# Quick look at what's inside
print(f'\n  Region labels inside box: {df_plastic_filtered["Region"].value_counts().to_dict()}')
print(f'  ⚠️  NOTE: Region labels are unreliable — "Arctic Ocean" records appear')
print(f'  at coordinates clearly in the Arabian Sea / Bay of Bengal.')
print(f'  We are using coordinates only, NOT the Region label, for all spatial work.')

In [ ]:
# Apply to primary species dataset
df_species_filtered = apply_bbox_filter(df_species, 'latitude', 'longitude', 'india_species')

print(f'\n  States inside box (top 5):')
print(df_species_filtered['state_province'].value_counts().head(5).to_string())
print(f'\n  Taxonomic groups:')
print(df_species_filtered['taxonomic_group'].value_counts().to_string())

In [ ]:
# Apply to realistic climate dataset
df_climate_filtered = apply_bbox_filter(df_climate, 'Latitude', 'Longitude', 'ocean_climate')

# IMPORTANT FINDING — this will be 0 records
print(f'\n  ⚠️  ISSUE FOUND: Zero records pass strict bbox filter!')
print(f'  Reason: Maldives is at Lat 3.21°N — just below our 5°N lower boundary.')
print(f'  Red Sea is at Lon 38.5°E — west of our 65°E boundary.')
print()
print('  Locations and their coordinates:')
print(df_climate.groupby('Location')[['Latitude','Longitude']].mean().round(2).to_string())
print()
print('  DECISION: Keeping this dataset unfiltered for now.')
print('  The Maldives (3.21°N) is biologically part of the Indian Ocean ecosystem.')
print('  Will discuss with mentor whether to extend LAT_MIN to 0°N for this dataset.')
print('  Logging this as Issue #1 in the metadata file.')

# Keep unfiltered version for now, flag it
df_climate_filtered = df_climate.copy()  # Use full dataset, flag for mentor review

In [ ]:
# Apply to microplastic datasets
df_micro_adv_filtered = apply_bbox_filter(df_micro_adv, 'Latitude', 'Longitude', 'micro_adventure')
df_micro_geo_filtered = apply_bbox_filter(df_micro_geo, 'Latitude', 'Longitude', 'micro_geomarine')
df_micro_sea_filtered = apply_bbox_filter(df_micro_sea, 'Latitude', 'Longitude', 'micro_sea')

print()
print('⚠️  All 3 microplastic datasets have near-zero records in our bounding box.')
print('These are Atlantic/Pacific focused surveys, not Indian Ocean surveys.')
print('Role: Supplementary context only. Will NOT be used in spatial join.')
print('Will use the "Normalized" column if referenced for density benchmarking.')

---
## Step 5: Filter Non-Spatial Datasets by Year (2015–2026)

Datasets without lat/lon get filtered by time window instead.

In [ ]:
YEAR_MIN = 2015
YEAR_MAX = 2026

# GlobalTemperatures — filter by year
df_temp['year'] = df_temp['dt'].dt.year
df_temp_filtered = df_temp[df_temp['year'].between(YEAR_MIN, YEAR_MAX)].copy()
print(f'GlobalTemperatures: {len(df_temp):,} → {len(df_temp_filtered):,} records  (year {YEAR_MIN}–{YEAR_MAX})')

# sea_level_co2 — already has year column
df_co2_filtered = df_co2[df_co2['year'].between(YEAR_MIN, YEAR_MAX)].copy()
print(f'sea_level_co2:      {len(df_co2):,} → {len(df_co2_filtered):,} records  (year {YEAR_MIN}–{YEAR_MAX})')

# Plastic_Waste_Around_the_World — no date column, keep as-is
df_world_filtered = df_world.copy()
print(f'plastic_world:      {len(df_world):,} records (no date column — using full dataset)')

# River data — filter to India
df_river_india = df_river[df_river['Country'] == 'India'].copy()
print(f'river_plastic:      {len(df_river):,} → {len(df_river_india):,} India records')

# Protected sites — already India only
df_protected_filtered = df_protected.copy()
print(f'protected_sites:    {len(df_protected):,} records (India only — no filter needed)')

---
## Step 6: Visualise What Made It Through the Filter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Plot 1: Plastic records ---
ax1 = axes[0]
ax1.scatter(
    df_plastic_filtered['Longitude'],
    df_plastic_filtered['Latitude'],
    c='crimson', s=35, alpha=0.7, zorder=4, label=f'Plastic ({len(df_plastic_filtered)} records)'
)
rect1 = mpatches.Rectangle(
    (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
    linewidth=2, edgecolor='darkred', facecolor='lightyellow', alpha=0.2, linestyle='--'
)
ax1.add_patch(rect1)
# Ocean zone labels
ax1.text(69, 16, 'Arabian\nSea',    fontsize=8, color='navy', style='italic', alpha=0.8)
ax1.text(84, 13, 'Bay of\nBengal', fontsize=8, color='navy', style='italic', alpha=0.8)
ax1.text(91, 11, 'Andaman\nSea',   fontsize=8, color='navy', style='italic', alpha=0.8)
# City dots
cities = {'Mumbai':(72.8,19.0),'Chennai':(80.3,13.0),'Kolkata':(88.4,22.6),
          'Kochi':(76.3,9.9),'Visakhapatnam':(83.3,17.7)}
for city, (lon, lat) in cities.items():
    ax1.plot(lon, lat, 'k^', ms=5)
    ax1.annotate(city, (lon, lat), textcoords='offset points', xytext=(4,4),
                 fontsize=6.5, color='black')
ax1.set_xlim(62, 98)
ax1.set_ylim(3, 32)
ax1.set_xlabel('Longitude (°E)')
ax1.set_ylabel('Latitude (°N)')
ax1.set_title('Plastic Pollution Records\nAfter Bounding Box Filter', fontsize=12)
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Species records ---
ax2 = axes[1]
colors = {'Aves':'steelblue','Mammalia':'darkorange','Reptilia':'green','Pisces':'purple'}
for group, color in colors.items():
    subset = df_species_filtered[df_species_filtered['taxonomic_group'] == group]
    ax2.scatter(
        subset['longitude'], subset['latitude'],
        c=color, s=5, alpha=0.25, label=f'{group} ({len(subset):,})'
    )
rect2 = mpatches.Rectangle(
    (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
    linewidth=2, edgecolor='darkred', facecolor='lightyellow', alpha=0.2, linestyle='--'
)
ax2.add_patch(rect2)
ax2.text(69, 16, 'Arabian\nSea',    fontsize=8, color='navy', style='italic', alpha=0.8)
ax2.text(84, 13, 'Bay of\nBengal', fontsize=8, color='navy', style='italic', alpha=0.8)
for city, (lon, lat) in cities.items():
    ax2.plot(lon, lat, 'k^', ms=5)
    ax2.annotate(city, (lon, lat), textcoords='offset points', xytext=(4,4),
                 fontsize=6.5, color='black')
ax2.set_xlim(62, 98)
ax2.set_ylim(3, 32)
ax2.set_xlabel('Longitude (°E)')
ax2.set_ylabel('Latitude (°N)')
ax2.set_title('Marine Species Observations\nAfter Bounding Box Filter', fontsize=12)
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle('May 20, 2026 — Spatial Boundary Filtering Results\nNorth Indian Ocean (Lat 5°N–30°N, Lon 65°E–95°E)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DIRS['visualizations'], 'Day2_bbox_filter_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Visualisation saved ✅')

---
## Step 7: Save Filtered Datasets to /data/processed/

In [ ]:
# Save all filtered datasets
save_map = {
    'plastic_filtered.csv':          df_plastic_filtered,
    'species_filtered.csv':          df_species_filtered,
    'climate_flagged.csv':           df_climate_filtered,   # unfiltered — flagged for mentor
    'global_temp_filtered.csv':      df_temp_filtered,
    'sea_level_co2_filtered.csv':    df_co2_filtered,
    'plastic_world.csv':             df_world_filtered,
    'river_india.csv':               df_river_india,
    'protected_sites.csv':           df_protected_filtered,
}

for filename, df in save_map.items():
    path = os.path.join(DIRS['processed'], filename)
    df.to_csv(path, index=False)
    print(f'Saved: {filename} ({len(df):,} rows) ✅')

---
## Step 8: Log Everything to Metadata

In [ ]:
import datetime

log_path = os.path.join(DIRS['metadata'], 'ingestion_log_may20.txt')

with open(log_path, 'w') as f:
    f.write('HCLTech Internship — Ingestion & Spatial Filter Log\n')
    f.write(f'Date: May 20, 2026 | Logged: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*60 + '\n\n')

    f.write('BOUNDING BOX APPLIED:\n')
    f.write(f'  Latitude  : {LAT_MIN}°N – {LAT_MAX}°N\n')
    f.write(f'  Longitude : {LON_MIN}°E – {LON_MAX}°E\n\n')

    f.write('FILTER RESULTS:\n')
    filter_results = [
        ('ocean_plastic',     len(df_plastic),    len(df_plastic_filtered)),
        ('india_species',     len(df_species),    len(df_species_filtered)),
        ('global_temp',       len(df_temp),       len(df_temp_filtered)),
        ('sea_level_co2',     len(df_co2),        len(df_co2_filtered)),
        ('river_india',       len(df_river),      len(df_river_india)),
        ('plastic_world',     len(df_world),      len(df_world_filtered)),
        ('protected_sites',   len(df_protected),  len(df_protected_filtered)),
    ]
    for name, before, after in filter_results:
        pct = round(after / before * 100, 1)
        f.write(f'  {name:<22}: {before:>6} → {after:>6} ({pct}% retained)\n')

    f.write('\nISSUES FLAGGED:\n')
    f.write('  Issue #1: realistic_ocean_climate_dataset — 0 records pass strict bbox.\n')
    f.write('            Maldives (Lat 3.21°N) is below LAT_MIN=5. Red Sea at Lon 38.5°E.\n')
    f.write('            Action: Kept full dataset. Mentor review needed.\n\n')
    f.write('  Issue #2: ocean_plastic Region label is unreliable.\n')
    f.write('            Records labelled Arctic/Atlantic appear inside Indian Ocean bbox.\n')
    f.write('            Action: Using raw coordinates only for all spatial work.\n\n')
    f.write('  Issue #3: india_cms year column has anomalous values (-980 to 9987).\n')
    f.write('            Action: Will fix in Week 2 cleaning.\n\n')
    f.write('  Issue #4: All 3 microplastic datasets have near-zero Indian Ocean coverage.\n')
    f.write('            Action: Supplementary context only. Not used in spatial join.\n')

print(f'Ingestion log saved to: {log_path} ✅')

# Print to screen
with open(log_path) as f:
    print(f.read())